In [1]:
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import json
import pysam
import cvxpy as cp

from utils import filename2ctype

%load_ext autoreload
%autoreload 2

## Data preparation

First we load the file containing the coverage per ctype per dmr.

In [2]:
df_coverage_per_ctype_per_dmr = pd.read_csv(
    Path("./data/region_coverage.csv")
)
df_coverage_per_ctype_per_dmr.columns = [col if col != "Ovary+Endom-Ep" else "Ovary-Ep" for col in df_coverage_per_ctype_per_dmr.columns]
atlas = pd.read_csv(Path("../../Data/Atlas.U25.l4.hg38.full.tsv"), sep="\t")
#add target column to df_coverage_per_ctype_per_dmr
map_region_key_to_target = atlas.set_index(["chr", "start", "end"])["target"].to_dict()
df_coverage_per_ctype_per_dmr["target"] = df_coverage_per_ctype_per_dmr.apply(
    lambda row: map_region_key_to_target.get((row["chr"], row["start"], row["end"])), axis=1
)

labels_dict = json.load(open(Path("../../App/labels_dict.json"), "r", encoding="utf-8"))
labels_dict = {int(k): v for k, v in labels_dict.items()}
n_ctypes = len(labels_dict)
labels_list = [labels_dict[i] for i in range(n_ctypes)]

Now we want to find the N ctypes such that:

$$
set = argmax_{set: |set|=N} \min_{i \in set}( \max_{dmr \in dmr_i} (\min_{j \in set}(coverage(dmr, j)))) 
$$

In other words, we want to find the set of N ctypes such that there is a least one dmr specific to each ctype in the set such that the minimum coverage of that dmr across the ctypes in the set is as high as possible.

For now we do it for 2 ctypes, where there is an efficient algorithm to do so.

In [3]:
matrix_max_coverage_per_dmr_ctype_per_ctype = np.zeros((n_ctypes, n_ctypes), dtype=int)
matrix_idx_max_coverage_per_dmr_ctype_per_ctype = np.zeros((n_ctypes, n_ctypes), dtype=int)
for i, ctype_i in enumerate(labels_list):
    tmp_df = df_coverage_per_ctype_per_dmr[df_coverage_per_ctype_per_dmr["target"] == ctype_i][labels_list]
    matrix_max_coverage_per_dmr_ctype_per_ctype[i] = tmp_df.max(axis=0).values.astype(int)
    matrix_idx_max_coverage_per_dmr_ctype_per_ctype[i] = tmp_df.idxmax(axis=0).values.astype(int)

# iterate over all pairs of ctypes and find the pair with the highest crossed minimum coverage
max_min_coverage = 0
ctype_pair = None
for i in range(n_ctypes):
    for j in range(i + 1, n_ctypes):
        min_coverage = min(matrix_max_coverage_per_dmr_ctype_per_ctype[i, j], matrix_max_coverage_per_dmr_ctype_per_ctype[j, i])
        if min_coverage > max_min_coverage:
            max_min_coverage = min_coverage
            ctype_pair = (labels_list[i], labels_list[j])

selected_dmr0 = matrix_idx_max_coverage_per_dmr_ctype_per_ctype[labels_list.index(ctype_pair[0]), labels_list.index(ctype_pair[1])]
selected_dmr1 = matrix_idx_max_coverage_per_dmr_ctype_per_ctype[labels_list.index(ctype_pair[1]), labels_list.index(ctype_pair[0])]
selected_dmrs = [selected_dmr0, selected_dmr1]
selected_dmrs_df = atlas.iloc[[selected_dmr0, selected_dmr1]]
print(f"Max min cross coverage: {max_min_coverage} for pair: {ctype_pair})")
print(selected_dmrs_df.to_string())

Max min cross coverage: 3231 for pair: ('Blood-T', 'Head-Neck-Ep'))
       chr      start        end  startCpG    endCpG        target                       name direction  Adipocytes  Bladder-Ep  Blood-B  Blood-Granul  Blood-Mono+Macro  Blood-NK  Blood-T  Bone-Osteob  Breast-Basal-Ep  Breast-Luminal-Ep  Colon-Ep  Colon-Fibro  Dermal-Fibro  Endothel  Epid-Kerat  Eryth-prog  Fallopian-Ep  Gallbladder  Gastric-Ep  Head-Neck-Ep  Heart-Cardio  Heart-Fibro  Kidney-Ep  Liver-Hep  Lung-Ep-Alveo  Lung-Ep-Bron  Neuron  Oligodend  Ovary-Ep  Pancreas-Acinar  Pancreas-Alpha  Pancreas-Beta  Pancreas-Delta  Pancreas-Duct  Megakaryocytes  Prostate-Ep  Skeletal-Musc  Small-Int-Ep  Smooth-Musc  Thyroid-Ep
561  chr11  118304916  118305780  17570807  17570821       Blood-T  chr11:118304916-118305780         U       0.014        0.01    0.004           0.0             0.040     0.273    0.987        0.037            0.005              0.000     0.021        0.093           0.0     0.005       0.000       

Now we retrieve all the reads from the selected ctype that overlap the selected DMRS

In [4]:
# for selected_ctype in ctype_pair:
#     sel_ctype_files = 

In [5]:
ROOT_DATA_DIR = Path("/staging/leuven/stg_00118/methylDL/data/loyfer2023/hg38/data/GSE186458")


In [9]:
(atlas["end"]-atlas["start"]).unique()

array([ 296,  585,  270,   77,  566,  249,  437,   68,  128,  174,  717,
        357, 1014,  291,  871,  641,  123,  754,  173,  208,   63,  316,
        257,  227,  348,  194,  683,  307,  584,  230,  490, 1273,  192,
        391,  893,  426,  106,  143,   57,  260,  200,   94,  508,  114,
        311,  636,   33,  417,  111,  356,  218,  729,  290,  279,  238,
        343,  232,  428,  241,  272,  152,  339,  533,  221,   11,  328,
        104,  329,  113,   95,  317,  286,   91,  499,  415,  996,  908,
       1094,  265,  164,  635,  166,  478,  411,   64,  242,  236,  304,
         99,  262,  182, 1181,  742,  528,  532,  292,  315,  346,  518,
        300,  129,  493, 1291,   62,  400,  154,  506,  321,  306, 1063,
         89,  395, 1059,  605,   83,  198,   78,  148,  531,  176,  190,
        149,   97,  618,  449,  897,   48,  244,  130,  595,  122,  167,
        183,  246,  523,  278,  318,  147,  217,   87,  127,  387,  444,
        155,  452,  231,  237,  782,  179,   79,  5

In [12]:
np.array(sorted((atlas["endCpG"]-atlas["startCpG"]).unique()))

array([ 5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21,
       22, 23, 24, 25, 26, 27, 29, 31, 34, 35, 37, 41, 46, 54])